## 11. Optional physical simulation and telemetry assimilation

The following self-contained helpers reproduce the project’s cellular-automata and latent Ensemble Kalman Filter components. They are optional experiments; they are not required to train or use the PINN checkpoint above.

In [ ]:
import numpy as np

class CellularAutomata3D:
    # States: 0 unburnt, 1 active smouldering, 2 burnt.
    def __init__(self, shape=(50, 50, 20), dx=0.1, dt=60.0):
        self.shape, self.dx, self.dt = shape, dx, dt
        self.grid = np.zeros(shape, dtype=np.int32)
        self.temp_grid = np.ones(shape) * 25.0

    def initialize_fire(self, x, y, z):
        self.grid[x, y, z], self.temp_grid[x, y, z] = 1, 300.0

    def step(self, moisture_grid, water_table_index):
        next_grid, next_temperature = self.grid.copy(), self.temp_grid.copy()
        x_dim, y_dim, z_dim = self.shape
        for x in range(1, x_dim - 1):
            for y in range(1, y_dim - 1):
                for z in range(1, z_dim - 1):
                    if z >= water_table_index:
                        next_grid[x, y, z], next_temperature[x, y, z] = 0, 20.0
                        continue
                    if self.grid[x, y, z] == 0:
                        active_neighbours = np.sum(self.grid[x-1:x+2, y-1:y+2, z-1:z+2] == 1)
                        ignition_probability = 0.3 / (1.0 + moisture_grid[x, y, z])
                        if active_neighbours and np.random.rand() < ignition_probability:
                            next_grid[x, y, z], next_temperature[x, y, z] = 1, 300.0
                    elif self.grid[x, y, z] == 1 and np.random.rand() < 0.05:
                        next_grid[x, y, z], next_temperature[x, y, z] = 2, 60.0
        self.grid, self.temp_grid = next_grid, next_temperature
        return self.grid, self.temp_grid

class LatentEnKF:
    def __init__(self, ensemble_size=50, latent_dim=10, obs_dim=5):
        self.M, self.d, self.p = ensemble_size, latent_dim, obs_dim
        self.ensemble = np.random.randn(self.d, self.M)

    def forecast(self, forward_model_fn):
        for index in range(self.M):
            self.ensemble[:, index] = forward_model_fn(self.ensemble[:, index])
        return self.ensemble

    def update(self, observations, observation_operator_h, r_covariance):
        mean = self.ensemble.mean(axis=1, keepdims=True)
        anomalies = self.ensemble - mean
        observed_ensemble = observation_operator_h @ self.ensemble
        observed_anomalies = observed_ensemble - observed_ensemble.mean(axis=1, keepdims=True)
        cross_covariance = anomalies @ observed_anomalies.T / (self.M - 1)
        observation_covariance = observed_anomalies @ observed_anomalies.T / (self.M - 1)
        gain = cross_covariance @ np.linalg.inv(observation_covariance + r_covariance)
        perturbation = np.random.multivariate_normal(np.zeros(self.p), r_covariance, size=self.M).T
        self.ensemble += gain @ (observations + perturbation - observed_ensemble)
        return self.ensemble

# Small reproducible examples. Replace these values with calibrated field assumptions.
ca = CellularAutomata3D(shape=(15, 15, 10))
ca.initialize_fire(7, 7, 1)
moisture_grid = np.full((15, 15, 10), 0.25)
ca_grid, _ = ca.step(moisture_grid, water_table_index=8)
print('Active CA cells after one step:', int((ca_grid == 1).sum()))

enkf = LatentEnKF(ensemble_size=20, latent_dim=4, obs_dim=2)
enkf.forecast(lambda state: 0.98 * state + np.random.normal(0, 0.02, size=state.shape))
updated = enkf.update(np.array([[0.2], [-0.1]]), np.random.randn(2, 4), np.eye(2) * 0.05)
print('Updated EnKF state shape:', updated.shape)

# EmberBrain: Colab training and inference

This is a **self-contained** notebook: upload it to Google Colab and run cells from top to bottom. It does not import or run `train.py`, `inference.py`, or any other repository module.

The preferred input is a long, spatial telemetry CSV with `x_m`, `y_m`, `z_m`, `time_s`, `temperature_c`, `oxygen_fraction`, and `water_table_m`. The notebook can also convert the current wide format (`temp_5`, `temp_15`, `temp_30`, `temp_45`) for a smoke test. A legacy file has no measured oxygen or node coordinates, so its resulting model is not suitable for deployment.

## 1. Runtime setup

Colab already includes PyTorch, NumPy, pandas, and matplotlib. This cell makes runs repeatable and selects a GPU when Colab provides one.

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Upload telemetry

Run the next cell and choose your `telemetry.csv`. If you skip the upload, the notebook creates a clearly labelled synthetic dataset only to verify the workflow.

In [ ]:
# In Google Colab, uncomment these two lines to upload your CSV.
# from google.colab import files
# uploaded = files.upload()

# Set this to the uploaded filename, for example: 'telemetry.csv'.
DATA_PATH = 'telemetry.csv'

# The output model is written to the current Colab session.
MODEL_PATH = 'emberbrain_pinn_colab.pt'

## 3. Data preparation

Coordinate convention: `z_m` is a positive depth below the peat surface. `water_table_m` is a negative depth below the surface, so `-0.15` means the water table is 15 cm down. The PINN treats `z_m <= abs(water_table_m)` as unsaturated peat.

Preferred CSV columns:

```text
timestamp,node_id,x_m,y_m,z_m,time_s,temperature_c,oxygen_fraction,co_ppm,co2_ppm,ch4_ppm,moisture_pct,water_table_m
```

In [ ]:
INPUT_COLUMNS = ['x_m', 'y_m', 'z_m', 'time_s']
TARGET_COLUMNS = ['temperature_c', 'oxygen_fraction']
REQUIRED_EXTENDED = {'timestamp', 'node_id', *INPUT_COLUMNS, *TARGET_COLUMNS, 'water_table_m'}
LEGACY_TEMP_DEPTHS_M = {'temp_5': 0.05, 'temp_15': 0.15, 'temp_30': 0.30, 'temp_45': 0.45}

def make_synthetic_extended_data(path='synthetic_telemetry.csv', n_times=120):
    """Creates sample data for testing the notebook only; replace it with field data."""
    rows = []
    start = pd.Timestamp('2026-07-20T00:00:00Z')
    for step in range(n_times):
        time_s = step * 300.0
        hotspot = 55 * np.exp(-((time_s - 18000) / 9000) ** 2)
        for z_m in (0.05, 0.15, 0.30, 0.45):
            temperature_c = 25 + hotspot * np.exp(-z_m * 3.2) + np.random.normal(0, 0.4)
            oxygen_fraction = max(0.03, 0.209 - hotspot / 500 * np.exp(-z_m * 2.0))
            rows.append({
                'timestamp': start + pd.Timedelta(seconds=time_s), 'node_id': 'demo_01',
                'x_m': 2.5, 'y_m': 1.75, 'z_m': z_m, 'time_s': time_s,
                'temperature_c': temperature_c, 'oxygen_fraction': oxygen_fraction,
                'co_ppm': 1.5 + hotspot / 10, 'co2_ppm': 402 + hotspot * 2,
                'ch4_ppm': 2.0, 'moisture_pct': 45 - hotspot / 5, 'water_table_m': -0.35,
            })
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def legacy_wide_to_long(df, node_coordinates=None):
    """Converts the repository's original wide sensor CSV to the extended schema.
    It supplies 0, 0 coordinates and ambient O2 because legacy data lacks them."""
    required = {'timestamp', 'node_id', *LEGACY_TEMP_DEPTHS_M, 'co', 'co2', 'ch4', 'moisture', 'water_table'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'CSV is neither extended nor valid legacy telemetry. Missing: {sorted(missing)}')
    result = df.melt(
        id_vars=['timestamp', 'node_id', 'co', 'co2', 'ch4', 'moisture', 'water_table'],
        value_vars=list(LEGACY_TEMP_DEPTHS_M), var_name='depth_column', value_name='temperature_c'
    )
    result['z_m'] = result['depth_column'].map(LEGACY_TEMP_DEPTHS_M)
    coordinates = node_coordinates or {}
    result[['x_m', 'y_m']] = result['node_id'].map(lambda node: coordinates.get(node, (0.0, 0.0))).apply(pd.Series)
    result['oxygen_fraction'] = 0.209  # Placeholder: collect measured O2 for a real model.
    result['co_ppm'] = result['co']
    result['co2_ppm'] = result['co2']
    result['ch4_ppm'] = result['ch4']
    result['moisture_pct'] = result['moisture']
    result['water_table_m'] = result['water_table'] / 100.0  # legacy values are centimetres
    return result.drop(columns=['depth_column', 'co', 'co2', 'ch4', 'moisture', 'water_table'])

def prepare_telemetry(path):
    raw = pd.read_csv(path)
    if REQUIRED_EXTENDED.issubset(raw.columns):
        df = raw.copy()
        source_format = 'extended'
    else:
        df = legacy_wide_to_long(raw)
        source_format = 'legacy-wide (converted; coordinates and O2 are placeholders)'

    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    for col in [*INPUT_COLUMNS, *TARGET_COLUMNS, 'water_table_m']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['timestamp', 'node_id', *INPUT_COLUMNS, *TARGET_COLUMNS, 'water_table_m']).copy()
    df = df.sort_values(['node_id', 'timestamp', 'z_m']).reset_index(drop=True)

    # A valid extended file may omit a useful elapsed time value; derive it per node if needed.
    if (df['time_s'] < 0).any() or df['time_s'].nunique() < 2:
        df['time_s'] = df.groupby('node_id')['timestamp'].transform(lambda values: (values - values.min()).dt.total_seconds())

    unique_times = df[['node_id', 'timestamp']].drop_duplicates().shape[0]
    if unique_times < 20:
        print(f'WARNING: only {unique_times} unique node/timestamp observations. Training will run, but will overfit and is not deployable.')
    print(f'Loaded {len(df):,} depth measurements from {source_format}.')
    return df

if not Path(DATA_PATH).exists():
    DATA_PATH = make_synthetic_extended_data()
    print(f'No uploaded file found. Using synthetic test data: {DATA_PATH}')

telemetry = prepare_telemetry(DATA_PATH)
display(telemetry.head())
print(telemetry[INPUT_COLUMNS + TARGET_COLUMNS + ['water_table_m']].describe().T)

## 4. Optional quality check

Inspect coverage before training. Large gaps, a single operating condition, or unverified fire labels produce a model that may fit the CSV but cannot generalize.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for depth, group in telemetry.groupby('z_m'):
    axes[0].plot(group['timestamp'], group['temperature_c'], '.', ms=3, label=f'{depth:.2f} m')
axes[0].set(title='Temperature by depth', xlabel='Time', ylabel='Temperature (°C)')
axes[0].legend(title='Depth')
axes[1].scatter(telemetry['moisture_pct'] if 'moisture_pct' in telemetry else 0, telemetry['temperature_c'], s=8, alpha=0.6)
axes[1].set(title='Temperature versus moisture', xlabel='Moisture (%)', ylabel='Temperature (°C)')
plt.tight_layout()

## 5. Telemetry features and fire-stage helpers

These are the repository's derived indicators, included here directly for exploratory analysis and alarm prototyping. The PINN below uses the spatial, time, water-table, temperature, and O₂ fields.

In [ ]:
def add_telemetry_features(df):
    result = df.copy()
    if {'co2_ppm', 'co_ppm'}.issubset(result):
        result['ratio_co2_co'] = result['co2_ppm'] / (result['co_ppm'] + 1e-5)
    result['temperature_dt_c_per_step'] = result.groupby(['node_id', 'z_m'])['temperature_c'].diff().fillna(0.0)
    result['temperature_ema_short'] = result.groupby(['node_id', 'z_m'])['temperature_c'].transform(lambda values: values.ewm(span=6, adjust=False).mean())
    result['temperature_ema_long'] = result.groupby(['node_id', 'z_m'])['temperature_c'].transform(lambda values: values.ewm(span=288, adjust=False).mean())
    result['temperature_deviation'] = result['temperature_c'] - result['temperature_ema_long']
    return result

def classify_fire_stage(temp_5, temp_15, co_ppm, ratio_co2_co, moisture_pct, water_table_m):
    # 0 normal, 1 monitor, 2 suspicious, 3 active-smouldering warning
    if water_table_m >= 0:
        return 0
    heating = temp_5 > 45.0 or temp_15 > 40.0
    gas_anomalous = co_ppm > 5.0 and ratio_co2_co < 50.0
    dry = moisture_pct < 25.0
    if heating and gas_anomalous:
        return 3
    if heating or (gas_anomalous and dry):
        return 2
    return 1 if temp_5 > 35.0 or co_ppm > 3.0 else 0

telemetry_features = add_telemetry_features(telemetry)
display(telemetry_features[['timestamp', 'node_id', 'z_m', 'temperature_c', 'temperature_dt_c_per_step', 'temperature_deviation']].head())

## 6. PINN surrogate

This network maps `(x, y, z, time)` to `(temperature, oxygen)`. The residual is a stable, nondimensional heat-equation constraint. It is a surrogate model, not a substitute for calibrated peat combustion parameters.

In [ ]:
class PINNSurrogate(nn.Module):
    def __init__(self, input_mean, input_std, target_mean, target_std, hidden_layers=(64, 64, 64)):
        super().__init__()
        self.register_buffer('input_mean', torch.as_tensor(input_mean, dtype=torch.float32))
        self.register_buffer('input_std', torch.as_tensor(input_std, dtype=torch.float32).clamp_min(1e-6))
        self.register_buffer('target_mean', torch.as_tensor(target_mean, dtype=torch.float32))
        self.register_buffer('target_std', torch.as_tensor(target_std, dtype=torch.float32).clamp_min(1e-6))
        layers, width = [], 4
        for hidden in hidden_layers:
            layers += [nn.Linear(width, hidden), nn.Tanh()]
            width = hidden
        layers.append(nn.Linear(width, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, y, z, t):
        coordinates = torch.cat([x, y, z, t], dim=1)
        normalized = (coordinates - self.input_mean) / self.input_std
        return self.net(normalized) * self.target_std + self.target_mean

    def heat_residual(self, x, y, z, t, water_table_m):
        # Derivatives are with respect to physical (metre, second) coordinates.
        x, y, z, t = [value.detach().clone().requires_grad_(True) for value in (x, y, z, t)]
        temperature, oxygen = self(x, y, z, t).split(1, dim=1)
        grad = lambda output, coordinate: torch.autograd.grad(output, coordinate, torch.ones_like(output), create_graph=True)[0]
        dT_dt = grad(temperature, t)
        laplacian = grad(grad(temperature, x), x) + grad(grad(temperature, y), y) + grad(grad(temperature, z), z)
        above_water = (z <= water_table_m.abs()).float()
        activation = torch.sigmoid((temperature - 45.0) / 5.0)
        reaction = above_water * oxygen.clamp_min(0.0) * activation
        thermal_diffusivity = 1e-7
        reaction_scale = 2e-3
        return dT_dt - thermal_diffusivity * laplacian - reaction_scale * reaction

    @torch.no_grad()
    def predict(self, x_m, y_m, z_m, time_s):
        values = [np.asarray(value, dtype=np.float32).reshape(-1, 1) for value in (x_m, y_m, z_m, time_s)]
        inputs = [torch.from_numpy(value).to(next(self.parameters()).device) for value in values]
        return self(*inputs).cpu().numpy()

## 7. Chronological training and validation

The final 20% of distinct timestamps is held out for validation. Do not use a random row split for time-series fire data. Reduce `EPOCHS` for a quick smoke test; increase it only after you have adequate field data.

In [ ]:
EPOCHS = 500
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
PHYSICS_WEIGHT = 0.01

ordered_times = np.sort(telemetry['timestamp'].unique())
split_time = ordered_times[max(1, int(len(ordered_times) * 0.8)) - 1]
train_df = telemetry[telemetry['timestamp'] <= split_time].copy()
val_df = telemetry[telemetry['timestamp'] > split_time].copy()
if val_df.empty:
    raise ValueError('Need at least two distinct timestamps to create a chronological validation split.')

input_mean = train_df[INPUT_COLUMNS].mean().to_numpy(np.float32)
input_std = train_df[INPUT_COLUMNS].std().fillna(1.0).replace(0, 1.0).to_numpy(np.float32)
target_mean = train_df[TARGET_COLUMNS].mean().to_numpy(np.float32)
target_std = train_df[TARGET_COLUMNS].std().fillna(1.0).replace(0, 1.0).to_numpy(np.float32)

def tensors_from_frame(frame):
    x = torch.tensor(frame[INPUT_COLUMNS].to_numpy(np.float32))
    y = torch.tensor(frame[TARGET_COLUMNS].to_numpy(np.float32))
    water_table = torch.tensor(frame[['water_table_m']].to_numpy(np.float32))
    return TensorDataset(x, y, water_table)

train_loader = DataLoader(tensors_from_frame(train_df), batch_size=min(BATCH_SIZE, len(train_df)), shuffle=True)
val_loader = DataLoader(tensors_from_frame(val_df), batch_size=min(BATCH_SIZE, len(val_df)), shuffle=False)

model = PINNSurrogate(input_mean, input_std, target_mean, target_std).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=30)

best_state, best_val = None, float('inf')
history = {'train': [], 'val': [], 'data': [], 'physics': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = data_total = physics_total = count = 0
    for coordinates, targets, water_table in train_loader:
        coordinates, targets, water_table = coordinates.to(DEVICE), targets.to(DEVICE), water_table.to(DEVICE)
        x, y, z, t = coordinates.split(1, dim=1)
        prediction = model(x, y, z, t)
        data_loss = ((prediction - targets) / model.target_std).pow(2).mean()
        physics_loss = model.heat_residual(x, y, z, t, water_table).pow(2).mean()
        loss = data_loss + PHYSICS_WEIGHT * physics_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_size = len(coordinates)
        total += loss.item() * batch_size
        data_total += data_loss.item() * batch_size
        physics_total += physics_loss.item() * batch_size
        count += batch_size

    model.eval()
    val_total, val_count = 0.0, 0
    with torch.no_grad():
        for coordinates, targets, _ in val_loader:
            coordinates, targets = coordinates.to(DEVICE), targets.to(DEVICE)
            prediction = model(*coordinates.split(1, dim=1))
            val_total += (((prediction - targets) / model.target_std).pow(2).mean().item()) * len(coordinates)
            val_count += len(coordinates)
    train_loss, val_loss = total / count, val_total / val_count
    scheduler.step(val_loss)
    for key, value in [('train', train_loss), ('val', val_loss), ('data', data_total / count), ('physics', physics_total / count)]:
        history[key].append(value)
    if val_loss < best_val:
        best_val = val_loss
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    if epoch == 1 or epoch % 50 == 0:
        print(f'Epoch {epoch:4d}/{EPOCHS} | train={train_loss:.5f} | val={val_loss:.5f} | data={data_total/count:.5f} | physics={physics_total/count:.8f}')

model.load_state_dict(best_state)
print(f'Loaded best validation checkpoint (normalized validation loss: {best_val:.5f}).')

## 8. Evaluate the trained model

A small validation loss only means the held-out rows resemble the training period. Use separate field campaigns and confirmed event outcomes to establish real-world accuracy.

In [ ]:
plt.figure(figsize=(9, 4))
plt.semilogy(history['train'], label='Train total')
plt.semilogy(history['val'], label='Validation data')
plt.semilogy(history['data'], '--', label='Train data')
plt.xlabel('Epoch')
plt.ylabel('Normalized loss')
plt.title('Training history')
plt.legend()
plt.show()

val_predictions = model.predict(*(val_df[column].to_numpy() for column in INPUT_COLUMNS))
temperature_mae = np.mean(np.abs(val_predictions[:, 0] - val_df['temperature_c'].to_numpy()))
oxygen_mae = np.mean(np.abs(val_predictions[:, 1] - val_df['oxygen_fraction'].to_numpy()))
print(f'Validation temperature MAE: {temperature_mae:.3f} °C')
print(f'Validation oxygen MAE: {oxygen_mae:.5f} fraction')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(val_df['temperature_c'], val_predictions[:, 0], alpha=0.7)
axes[0].set(xlabel='Observed temperature (°C)', ylabel='Predicted temperature (°C)', title='Temperature validation')
axes[1].scatter(val_df['oxygen_fraction'], val_predictions[:, 1], alpha=0.7)
axes[1].set(xlabel='Observed O₂ fraction', ylabel='Predicted O₂ fraction', title='Oxygen validation')
for axis in axes:
    lower = min(axis.get_xlim()[0], axis.get_ylim()[0])
    upper = max(axis.get_xlim()[1], axis.get_ylim()[1])
    axis.plot([lower, upper], [lower, upper], 'k--', lw=1)
plt.tight_layout()

## 9. Inference: predict a temperature field

Choose a time and depth within the observed range. Extrapolating beyond the monitored area or time window is not validated.

In [ ]:
PREDICT_TIME_S = float(telemetry['time_s'].median())
PREDICT_DEPTH_M = 0.15
GRID_SIZE = 60

x_values = np.linspace(telemetry['x_m'].min(), telemetry['x_m'].max(), GRID_SIZE)
y_values = np.linspace(telemetry['y_m'].min(), telemetry['y_m'].max(), GRID_SIZE)
# A single fixed node has no spatial extent; use a small display area only for visualization.
if np.ptp(x_values) == 0:
    x_values = x_values[0] + np.linspace(-0.5, 0.5, GRID_SIZE)
if np.ptp(y_values) == 0:
    y_values = y_values[0] + np.linspace(-0.5, 0.5, GRID_SIZE)
grid_x, grid_y = np.meshgrid(x_values, y_values)
field_prediction = model.predict(grid_x.ravel(), grid_y.ravel(), np.full(grid_x.size, PREDICT_DEPTH_M), np.full(grid_x.size, PREDICT_TIME_S))
temperature_field = field_prediction[:, 0].reshape(grid_x.shape)

plt.figure(figsize=(7, 5))
image = plt.pcolormesh(grid_x, grid_y, temperature_field, shading='auto', cmap='hot')
plt.colorbar(image, label='Predicted temperature (°C)')
plt.scatter(telemetry['x_m'], telemetry['y_m'], c='cyan', s=15, edgecolors='black', label='Sensor measurements')
plt.title(f'Predicted temperature at z={PREDICT_DEPTH_M:.2f} m, t={PREDICT_TIME_S:.0f} s')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.legend()
plt.show()

## 10. Save and download the trained model

The checkpoint includes model weights and its normalization values. Keep it with a copy of the telemetry schema and sensor metadata.

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'input_columns': INPUT_COLUMNS,
    'target_columns': TARGET_COLUMNS,
    'best_validation_loss': best_val,
    'coordinate_convention': 'z_m positive down; water_table_m negative below surface',
}
torch.save(checkpoint, MODEL_PATH)
print(f'Saved model to {MODEL_PATH}')

# In Google Colab, uncomment to download the checkpoint to your computer.
# from google.colab import files
# files.download(MODEL_PATH)

## Data readiness checklist

- Use at least 20 distinct time points for a runnable validation split; use weeks of observations for a meaningful model.
- Record coordinates for every node and measured O₂ if you want valid spatial/O₂ predictions.
- Include normal, dry, warming, and confirmed smouldering conditions.
- Split complete later time windows or field campaigns into validation/test sets; never randomize adjacent readings across splits.
- Treat legacy-wide CSV training as a pipeline test only, because it supplies placeholder coordinates and oxygen values.